In [8]:
import pandas as pd
import requests

In [9]:
# USGS Earthquake API endpoint
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Query parameters to retrieve 1 earthquake in GeoJSON format
params = {
    "format": "geojson",
    "eventtype": "earthquake",
    "minmagnitude": 2.0,
    "starttime": "2026-10-01T00:00:00",
    "endtime": "2026-10-31T23:59:59",
    "limit": 20000
}

In [10]:
# Send GET request
response = requests.get(url, params=params, timeout=10)
data = response.json()
response

<Response [200]>

In [11]:
# Read the GeoJSON features into a pandas DataFrame
df = pd.json_normalize(data["features"])

# Extract longitude, latitude, and depth from geometry.coordinates
df[["longitude", "latitude", "depth_km"]] = pd.DataFrame(
    df["geometry.coordinates"].tolist(), index=df.index
)

# Clean up column prefixes (e.g., 'properties.mag' -> 'mag')
df.columns = [col.replace("properties.", "") for col in df.columns]

print(f"Retrieved {len(df)} earthquakes.")
df.head()

Retrieved 3812 earthquakes.


,type,id,mag,place,time,updated,tz,url,detail,felt,...,rms,gap,magType,type,title,geometry.type,geometry.coordinates,longitude,latitude,depth_km
0,Feature,us7000tule,4.80,"199 km NW of Port-Olry, Vanuatu",1793491075354,1793492226040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.71,110,mb,earthquake,"M 4.8 - 199 km NW of Port-Olry, Vanuatu",Point,"[165.6627, -13.8748, 10]",165.662700,-13.874800,10.00
1,Feature,uu80162266,2.18,"62 km SW of Dugway, Utah",1793490900380,1793647273230,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.22,134,ml,earthquake,"M 2.2 - 62 km SW of Dugway, Utah",Point,"[-113.2995, 39.8575, 4]",-113.299500,39.857500,4.00
2,Feature,aka2026vpmcdb,2.00,"62 km SE of Kokhanok, Alaska",1793490188184,1793607665202,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.20,56,ml,earthquake,"M 2.0 - 62 km SE of Kokhanok, Alaska",Point,"[-153.998, 59.041, 111.9]",-153.998000,59.041000,111.90
3,Feature,hv75066981,2.20,"7 km SW of Volcano, Hawaii",1793489463070,1793516180040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.11,51,ml,earthquake,"M 2.2 - 7 km SW of Volcano, Hawaii",Point,"[-155.2785, 19.3923333333333, 1.77]",-155.278500,19.392333,1.77
4,Feature,nc75460612,2.76,"52 km W of Petrolia, CA",1793488907920,1794716207040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,0.27,238,md,earthquake,"M 2.8 - 52 km W of Petrolia, CA",Point,"[-124.892833333333, 40.3565, 16.61]",-124.892833,40.356500,16.61


In [12]:
# 1. Convenience sample: first 100 rows returned by the API's default sort order (most recent first)
convenience_df = df.iloc[:100].copy()

In [13]:
# 2. Simple Random Sample (SRS): n = 100 with random_state = 42
srs_df = df.sample(n=100, random_state=42).copy()

In [14]:
# 3. Compare mean magnitude and time span of both samples (x-bar) against the population (mu)
def summarize_dataset(data, label):
    mean_mag = data["mag"].mean()
    start_dt = data["datetime"].min()
    end_dt = data["datetime"].max()
    time_span = end_dt - start_dt
    return {
        "Dataset": label,
        "Count (n)": len(data),
        "Mean Magnitude": round(mean_mag, 4),
        "Earliest Event (UTC)": start_dt,
        "Latest Event (UTC)": end_dt,
        "Time Span": time_span
    }

In [15]:
summary_df = pd.DataFrame([
    summarize_dataset(df, "Population (mu)"),
    summarize_dataset(convenience_df, "Convenience Sample (first 100)"),
    summarize_dataset(srs_df, "Simple Random Sample (n=100, seed=42)")
])

mu_mag = df["mag"].mean()
summary_df["Mag Diff from mu"] = (summary_df["Mean Magnitude"] - mu_mag).round(4)

summary_df

KeyError: 'datetime'